In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Input dan output Estimator

<Accordion>
<AccordionItem title="Versi pakej">

Kod di halaman ini dibangunkan menggunakan keperluan berikut.
Kami syorkan menggunakan versi ini atau yang lebih baru.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Halaman ini memberikan gambaran keseluruhan input dan output primitif Estimator Qiskit Runtime, yang melaksanakan beban kerja pada sumber pengiraan IBM Quantum&reg;. Estimator membolehkan kamu menentukan beban kerja vektorkan dengan cekap menggunakan struktur data yang dipanggil [**Blok Bersatu Primitif (PUB)**](/guides/primitive-input-output#pubs). Ia digunakan sebagai input kepada kaedah [`run()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2#run) untuk primitif Estimator, yang melaksanakan beban kerja yang ditentukan sebagai kerja. Kemudian, selepas kerja selesai, keputusan dikembalikan dalam format yang bergantung pada kedua-dua PUB yang digunakan serta pilihan runtime yang ditentukan dari primitif.
## Input
Setiap PUB berada dalam format ini:

(`<Circuit tunggal>`, `<satu atau lebih boleh pantau>`, `<nilai parameter pilihan satu atau lebih>`, `<ketepatan pilihan>`),

`nilai parameter` pilihan boleh berupa senarai atau parameter tunggal. Elemen dari boleh pantau dan nilai parameter digabungkan dengan mengikuti peraturan penyiaran NumPy seperti yang diterangkan dalam topik [Input dan output primitif](primitive-input-output#broadcasting-rules), dan satu anggaran nilai jangkaan dikembalikan untuk setiap elemen bentuk yang disiarkan.

> **Note:** Jika input mengandungi ukuran, ia diabaikan.

Untuk primitif Estimator, PUB boleh mengandungi paling banyak empat nilai:
- Satu `QuantumCircuit` tunggal, yang mungkin mengandungi satu atau lebih objek [`Parameter`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Parameter)
- Senarai satu atau lebih boleh pantau, yang menentukan nilai jangkaan yang perlu dianggarkan, disusun ke dalam tatasusunan (contohnya, boleh pantau tunggal yang diwakili sebagai tatasusunan 0-d, senarai boleh pantau sebagai tatasusunan 1-d, dan sebagainya). Data boleh berada dalam mana-mana format `ObservablesArrayLike` seperti `Pauli`, `SparsePauliOp`, `PauliList`, atau `str`.
   > **Note:** - Boleh pantau yang bertukar ganti **dalam PUB yang sama** dikumpulkan bersama menggunakan [kaedah ini](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.PauliList#group_qubit_wise_commuting).
>    - Boleh pantau yang bertukar ganti dalam PUB yang berbeza, walaupun mereka mempunyai Circuit yang sama, tidak dianggarkan menggunakan ukuran yang sama. Setiap PUB mewakili asas berbeza untuk ukuran, dan oleh itu, ukuran berasingan diperlukan untuk setiap PUB.
>    - Untuk memastikan boleh pantau yang bertukar ganti dianggarkan menggunakan ukuran yang sama, kumpulkan mereka dalam PUB yang sama.
- Koleksi nilai parameter untuk mengikat Circuit. Ini boleh ditentukan sebagai objek tunggal seperti tatasusunan di mana indeks terakhir adalah untuk objek `Parameter` Circuit atau dihilangkan (atau setaranya, ditetapkan kepada `None`) jika Circuit tidak mempunyai objek `Parameter`.
- (Secara pilihan) Ketepatan sasaran untuk nilai jangkaan yang perlu dianggarkan
---

Kod berikut menunjukkan set input vektorkan contoh untuk primitif `Estimator` dan melaksanakannya pada Backend IBM&reg; sebagai satu objek `RuntimeJobV2`.

In [1]:
from qiskit.circuit import (
    Parameter,
    QuantumCircuit,
)
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp

from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    EstimatorV2 as Estimator,
)

import numpy as np

# Instantiate runtime service and get
# the least busy backend
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Define a circuit with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.rz(Parameter("b"), 0)
circuit.cx(0, 1)
circuit.h(0)

# Transpile the circuit
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)
layout = transpiled_circuit.layout

# Now define a sweep over parameter values, the last axis of dimension 2 is
# for the two parameters "a" and "b"
params = np.vstack(
    [
        np.linspace(-np.pi, np.pi, 100),
        np.linspace(-4 * np.pi, 4 * np.pi, 100),
    ]
).T

# Define three observables. The inner length-1 lists cause this array of
# observables to have shape (3, 1), rather than shape (3,) if they were
# omitted.
observables = [
    [SparsePauliOp(["XX", "IY"], [0.5, 0.5])],
    [SparsePauliOp("XX")],
    [SparsePauliOp("IY")],
]
# Apply the same layout as the transpiled circuit.
observables = [
    [observable.apply_layout(layout) for observable in observable_set]
    for observable_set in observables
]

# Estimate the expectation value for all 300 combinations of observables
# and parameter values, where the pub result will have shape (3, 100).
#
# This shape is due to our array of parameter bindings having shape
# (100, 2), combined with our array of observables having shape (3, 1).
estimator_pub = (transpiled_circuit, observables, params)

# Instantiate the new Estimator object, then run the transpiled circuit
# using the set of parameters and observables.
estimator = Estimator(mode=backend)
job = estimator.run([estimator_pub])
result = job.result()

## Output
Selepas satu atau lebih PUB dihantar ke QPU untuk pelaksanaan dan kerja berjaya diselesaikan, data dikembalikan sebagai objek bekas [`PrimitiveResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult) yang diakses dengan memanggil kaedah `RuntimeJobV2.result()`.

`PrimitiveResult` mengandungi senarai iterable objek [`PubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult) yang mengandungi keputusan pelaksanaan untuk setiap PUB.

Setiap elemen senarai ini sepadan dengan setiap PUB yang dihantar kepada kaedah `run()` primitif (contohnya, kerja yang dihantar dengan 20 PUB akan mengembalikan objek `PrimitiveResult` yang mengandungi senarai 20 objek `PubResult`, satu sepadan dengan setiap PUB).

Setiap `PubResult` untuk primitif Estimator mengandungi sekurang-kurangnya tatasusunan nilai jangkaan (`PubResult.data.evs`) dan sisihan piawai yang berkaitan (sama ada `PubResult.data.stds` atau `PubResult.data.ensemble_standard_error` bergantung pada `resilience_level` yang digunakan), tetapi boleh mengandungi lebih banyak data bergantung pada pilihan pengurangan ralat yang ditentukan.

Setiap objek `PubResult` mempunyai atribut `data` dan `metadata`.
    - Atribut `data` adalah [`DataBin`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.DataBin) yang disesuaikan yang mengandungi nilai ukuran sebenar, sisihan piawai, dan sebagainya.
    - `DataBin` mempunyai pelbagai atribut bergantung pada bentuk atau struktur PUB yang berkaitan serta pilihan pengurangan ralat yang ditentukan oleh primitif yang digunakan untuk menghantar kerja (contohnya, [ZNE](/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) atau [PEC](/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec)).
    - Atribut `metadata` mengandungi maklumat tentang runtime dan pilihan pengurangan ralat yang digunakan (dijelaskan kemudian dalam bahagian [Metadata keputusan](#result-metadata) halaman ini).

Berikut adalah garis besar visual struktur data `PrimitiveResult` untuk output Estimator:

    ```
    └── PrimitiveResult
        ├── PubResult[0]
        │   ├── metadata
        │   └── data  ## In the form of a DataBin object
        │       ├── evs
        │       │   └── List of estimated expectation values in the shape
        |       |         specified by the first pub
        │       └── stds
        │           └── List of calculated standard deviations in the
        |                 same shape as above
        ├── PubResult[1]
        |   ├── metadata
        |   └── data  ## In the form of a DataBin object
        |       ├── evs
        |       │   └── List of estimated expectation values in the shape
        |       |        specified by the second pub
        |       └── stds
        |           └── List of calculated standard deviations in the
        |                same shape as above
        ├── ...
        ├── ...
        └── ...
    ```

Ringkasnya, satu kerja mengembalikan objek `PrimitiveResult` dan mengandungi senarai satu atau lebih objek `PubResult`. Objek `PubResult` ini kemudian menyimpan data ukuran untuk setiap PUB yang dihantar kepada kerja.

Petikan kod di bawah menerangkan format `PrimitiveResult` (dan `PubResult` yang berkaitan) untuk kerja yang dicipta di atas.

In [2]:
print(
    f"The result of the submitted job had {len(result)} PUBs and has a value:\n {result}\n"
)
print(
    f"The associated PubResult of this job has the following data bins:\n {result[0].data}\n"
)
print(f"And this DataBin has attributes: {result[0].data.keys()}")
print(
    "Recall that this shape is due to our array of parameter binding sets having shape (100, 2) -- where 2 is the\n\
         number of parameters in the circuit -- combined with our array of observables having shape (3, 1). \n"
)
with np.printoptions(threshold=200):
    print(
        f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
    )

The result of the submitted job had 1 PUB and has a value:
 PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(3, 100), dtype=float64>), stds=np.ndarray(<shape=(3, 100), dtype=float64>), ensemble_standard_error=np.ndarray(<shape=(3, 100), dtype=float64>), shape=(3, 100)), metadata={'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32})], metadata={'dynamical_decoupling': {'enable': False, 'sequence_type': 'XX', 'extra_slack_distribution': 'middle', 'scheduling_method': 'alap'}, 'twirling': {'enable_gates': False, 'enable_measure': True, 'num_randomizations': 'auto', 'shots_per_randomization': 'auto', 'interleave_randomizations': True, 'strategy': 'active-accum'}, 'resilience': {'measure_mitigation': True, 'zne_mitigation': False, 'pec_mitigation': False}, 'version': 2})

The associated PubResult of this job has the following data bins:
 DataBin(evs=np.ndarray(<shape=(3, 100), dtype=float64>), stds=np.ndarray(<shape=

#### How the Estimator primitive calculates error

In addition to the estimate of the mean of the observables passed in the input PUBs (the `evs` field of the `DataBin`), Estimator also attempts to deliver an estimate of the error associated with those expectation values. All Estimator queries will populate the `stds` field with a quantity like the standard error of the mean for each expectation value, but some error mitigation options produce additional information, such as `ensemble_standard_error`.

Consider a single observable $\mathcal{O}$. In the absence of [ZNE](/docs/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne), you can think of each shot of the Estimator execution as providing a point estimate of the expectation value $\langle \mathcal{O} \rangle$. If the pointwise estimates are in a vector `Os`, then the value returned in `ensemble_standard_error` is equivalent to the following (in which $\sigma_{\mathcal{O}}$ is the [standard deviation of the expectation value](/docs/api/qiskit/qiskit.primitives.BackendEstimatorV2) estimate and $N_{shots}$ is the number of shots):

$$\frac{ \sigma_{\mathcal{O}} }{ \sqrt{N_{shots}} },$$

which treats all shots as part of a single ensemble. If you requested gate [twirling](/docs/guides/error-mitigation-and-suppression-techniques#pauli-twirling) (`twirling.enable_gates = True`), you can sort the pointwise estimates of $\langle \mathcal{O} \rangle$ into sets that share a common twirl. Call these sets of estimates `O_twirls`, and there are `num_randomizations` (number of twirls) of them. Then `stds` is the standard error of the mean of `O_twirls`, as in

$$\frac{ \sigma_{\mathcal{O}} }{ \sqrt{N_{twirls}} },$$

where $\sigma_{\mathcal{O}}$ is the standard deviation of `O_twirls` and $N_{twirls}$ is the number of twirls. When you do not enable twirling, `stds` and `ensemble_standard_error` are equal.

If you enable ZNE, then the `stds` described above become weights in a non-linear regression to an extrapolator model. What finally gets returned in the `stds` in this case is the uncertainty of the fit model evaluated at a noise factor of zero. When there is a poor fit, or large uncertainty in the fit, the reported `stds` can become very large. When ZNE is enabled, `pub_result.data.evs_noise_factors` and `pub_result.data.stds_noise_factors` are also populated, so that you can do your own extrapolation.

## Result metadata

In addition to the execution results, both the `PrimitiveResult` and `PubResult` objects contain a metadata attribute about the job that was submitted. The metadata containing information for all submitted PUBs (such as the various [runtime options](/docs/api/qiskit-ibm-runtime/options) available) can be found in the `PrimitiveResult.metatada`, while the metadata specific to each PUB is found in `PubResult.metadata`.

<Admonition type="note">
In the metadata field, primitive implementations can return any information about execution that is relevant to them, and there are no key-value pairs that are guaranteed by the base primitive. Thus, the returned metadata might be different in different primitive implementations.
</Admonition>

In [3]:
# Print out the results metadata
print("The metadata of the PrimitiveResult is:")
for key, val in result.metadata.items():
    print(f"'{key}' : {val},")

print("\nThe metadata of the PubResult result is:")
for key, val in result[0].metadata.items():
    print(f"'{key}' : {val},")

The metadata of the PrimitiveResult is:
'dynamical_decoupling' : {'enable': False, 'sequence_type': 'XX', 'extra_slack_distribution': 'middle', 'scheduling_method': 'alap'},
'twirling' : {'enable_gates': False, 'enable_measure': True, 'num_randomizations': 'auto', 'shots_per_randomization': 'auto', 'interleave_randomizations': True, 'strategy': 'active-accum'},
'resilience' : {'measure_mitigation': True, 'zne_mitigation': False, 'pec_mitigation': False},
'version' : 2,

The metadata of the PubResult result is:
'shots' : 4096,
'target_precision' : 0.015625,
'circuit_metadata' : {},
'resilience' : {},
'num_randomizations' : 32,
